# Predict statistics labels with a fine‑tuned checkpoint

This notebook loads a fine‑tuned Transformer model from a saved checkpoint and predicts probabilities for the full unlabeled set and a set of conflicting descriptions. Configure the checkpoint and file paths in the config cell.

In [ ]:
# Configuration
MODEL_ID = "xlm-roberta-large"  # must match the fine-tuned checkpoint base
MODEL_CHECKPOINT = "./drive/MyDrive/Colab Notebooks/models/results/large-checkpoint-2169"  # <- adjust

# Data locations 
PATH_STAT_TO_MINE = "./drive/MyDrive/Colab Notebooks/stat_to_mine.feather"
PATH_CONFLICTING   = "./drive/MyDrive/Colab Notebooks/conflicting_descr_stat.feather"

# Outputs
OUT_UNLABELED_FEATHER   = "./drive/MyDrive/Colab Notebooks/unlabeled_predicted_large.feather"
OUT_CONFLICTING_FEATHER = "./drive/MyDrive/Colab Notebooks/conflicting_stat_predicted_large.feather"

# Inference parameters
BATCH_SIZE = 128  # adjust for GPU memory
MAX_TOKEN_LENGTH = None  # None => model max
SEED = 42

In [ ]:
# Setup: install (if needed) and import libs
# If running in Colab, uncomment the pip installs below.
# !pip install -U transformers datasets accelerate scikit-learn plotly pyarrow

import os
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [ ]:
# Mount drive to import datasets
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Load data
stat_to_mine = pd.read_feather(PATH_STAT_TO_MINE)
stat_to_mine = stat_to_mine.dropna(subset=["text_mining_description"])  # safety

unlabeled = stat_to_mine[(stat_to_mine["is_statistics"] == False) & (stat_to_mine["is_mining"] == False)].copy()
print(f"Unlabeled rows: {len(unlabeled):,}")

conflicting_stats = pd.read_feather(PATH_CONFLICTING)
conflicting_stats = conflicting_stats.dropna(subset=["text_mining_description"]).copy()
print(f"Conflicting rows: {len(conflicting_stats):,}")

In [ ]:
# Tokenizer and tokenization helpers
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding=True,               # dynamic padding
        truncation=True,
        max_length=MAX_TOKEN_LENGTH
    )

# Build HF datasets
unlabeled_texts = unlabeled["text_mining_description"].astype(str).tolist()
conflicting_texts = conflicting_stats["text_mining_description"].astype(str).tolist()

unlabeled_ds = Dataset.from_dict({"text": unlabeled_texts}).map(tokenize_fn, batched=True)
conflicting_ds = Dataset.from_dict({"text": conflicting_texts}).map(tokenize_fn, batched=True)

In [ ]:
# Load fine‑tuned model from checkpoint for inference
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)

inference_args = TrainingArguments(
    output_dir="./tmp-results",  # no checkpointing for inference
    per_device_eval_batch_size=BATCH_SIZE,
    do_train=False,
    do_eval=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    logging_strategy="no",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=inference_args,
    tokenizer=tokenizer,
    data_collator=collator,
)

In [ ]:
# Predict unlabeled
pred_unlabeled = trainer.predict(unlabeled_ds)
probs_unlabeled = torch.softmax(torch.tensor(pred_unlabeled.predictions), dim=1)[:, 1].cpu().numpy()

unlabeled = unlabeled.copy()
unlabeled["probability_is_statistics"] = probs_unlabeled

print("Unlabeled prediction done.")
unlabeled.to_feather(OUT_UNLABELED_FEATHER)
print(f"Saved: {OUT_UNLABELED_FEATHER}")

In [ ]:
# Predict conflicting descriptions
pred_conflicting = trainer.predict(conflicting_ds)
probs_conflicting = torch.softmax(torch.tensor(pred_conflicting.predictions), dim=1)[:, 1].cpu().numpy()

conflicting_stats = conflicting_stats.copy()
conflicting_stats["probability_is_statistics"] = probs_conflicting

print("Conflicting prediction done.")
conflicting_stats.to_feather(OUT_CONFLICTING_FEATHER)
print(f"Saved: {OUT_CONFLICTING_FEATHER}")

In [ ]:
# Quick sanity checks
print(unlabeled[["probability_is_statistics"]].describe())
print(conflicting_stats[["probability_is_statistics"]].describe())

unlabeled.head(3), conflicting_stats.head(3)